In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report
# Load your imbalanced data into a Pandas DataFrame
df = pd.read_csv('liver.csv')


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Assuming 'data' is your DataFrame
label_encoder = LabelEncoder()

df['Gender'] = label_encoder.fit_transform(df['Gender'])
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import cross_val_score


# Assuming 'Dataset' is your target variable
X = df.drop("Dataset", axis=1)
y = df["Dataset"]

# Mapping numeric values for Gender (assuming 'Gender' is a binary variable)
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 2})


# Filling missing values with the median
X.fillna(X.median(), inplace=True)

# Splitting the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
# Save the trained model
import joblib
print(X_train)
joblib.dump(lr, 'liver_model.sav')
lr_score = lr.score(X_test, y_test)

# Random Forest Classifier - Hyperparameter tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(RandomForestClassifier(), param_grid, cv=5)
grid_search.fit(X_train, y_train)
best_rfc = grid_search.best_estimator_
best_rfc_score = best_rfc.score(X_test, y_test)

# Support Vector Machine (SVM)
svm = SVC()
svm.fit(X_train, y_train)
svm_score = svm.score(X_test, y_test)

# Stacking Classifier
estimators = [
    ('lr', LogisticRegression()),
    ('best_rfc', best_rfc),
    ('svm', SVC())
]

stacking_model = StackingClassifier(estimators=estimators, final_estimator=RandomForestClassifier())
stacking_model.fit(X_train, y_train)
stacking_score = stacking_model.score(X_test, y_test)

# Voting Classifier
voting_model = VotingClassifier(estimators=estimators, voting='hard')
voting_model.fit(X_train, y_train)
voting_score = voting_model.score(X_test, y_test)

# Model Comparison
model_names = ["Logistic Regression", "Random Forest Classifier", "Support Vector Machine", "Stacking Classifier", "Voting Classifier"]
model_scores = [lr_score, best_rfc_score, svm_score, stacking_score, voting_score]

plt.figure(figsize=(10, 5))
sns.barplot(x=model_names, y=model_scores, palette="Blues_r")
plt.ylabel("Model Accuracy")
plt.title("Model Comparison - Model Accuracy", fontsize=14, fontname="Helvetica", y=1.03)
plt.show()

# Confusion Matrix and Classification Report for Stacking Classifier
y_pred_stacking = stacking_model.predict(X_test)
cf_matrix_stacking = confusion_matrix(y_test, y_pred_stacking)
print("\nConfusion Matrix for Stacking Classifier:")
print(cf_matrix_stacking)
print("\nClassification Report for Stacking Classifier:")
print(classification_report(y_test, y_pred_stacking))

# Confusion Matrix and Classification Report for Voting Classifier
y_pred_voting = voting_model.predict(X_test)
cf_matrix_voting = confusion_matrix(y_test, y_pred_voting)
print("\nConfusion Matrix for Voting Classifier:")
print(cf_matrix_voting)
print("\nClassification Report for Voting Classifier:")
print(classification_report(y_test, y_pred_voting))

In [ ]:
import pickle
pickle.dump(stacking_model,open('liver.pkl','wb'))